# 30.8 - Support Vector Regression


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
df = sns.load_dataset('tips')

In [5]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [7]:
df.columns

Index(['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size'], dtype='object')

In [8]:
# Dependent and independent features
X = df[['tip', 'sex', 'smoker', 'day', 'time', 'size']]
y = df['total_bill']

In [9]:
## Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =train_test_split(X, y, test_size = 0.25, random_state = 42)

In [10]:
# Feature Encoding

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

In [11]:
X_train['smoker']=le1.fit_transform(X_train['smoker'])
X_train['sex']=le2.fit_transform(X_train['sex'])
X_train['time']=le3.fit_transform(X_train['time'])

In [12]:
X_train.head()

,tip,sex,smoker,day,time,size
115,3.50,0,0,Sun,0,2
181,5.65,1,1,Sun,0,2
225,2.50,0,1,Fri,1,2
68,2.01,1,0,Sat,0,2
104,4.08,0,0,Sat,0,2


In [13]:
X_test['smoker']=le1.transform(X_test['smoker'])
X_test['sex']=le2.transform(X_test['sex'])
X_test['time']=le3.transform(X_test['time'])

In [14]:
from math import remainder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers=[('onehot', OneHotEncoder(drop='first'), [3])], 
                      remainder='passthrough')

In [15]:
X_train = ct.fit_transform(X_train)

In [16]:
X_test = ct.transform(X_test)

In [17]:
# SVR
from sklearn.svm import SVR

svr = SVR()


In [18]:
svr.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [19]:
y_pred = svr.predict(X_test)

In [20]:
from sklearn.metrics import mean_absolute_error, r2_score

print("R2 Score:",r2_score(y_test, y_pred))
print("MAE:",mean_absolute_error(y_test, y_pred))

R2 Score: 0.49798620106004743
MAE: 4.463296539661224


In [27]:
# Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
  'C': [0.1, 1, 10, 100, 1000],
  'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
  'kernel': ['poly']
}


In [28]:
from tabnanny import verbose


grid = GridSearchCV(SVR(), param_grid, refit = True, verbose=3)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV 1/5] END .......C=0.1, gamma=1, kernel=poly;, score=0.263 total time=   0.0s
[CV 2/5] END ......C=0.1, gamma=1, kernel=poly;, score=-0.544 total time=   0.0s
[CV 3/5] END .......C=0.1, gamma=1, kernel=poly;, score=0.066 total time=   0.0s
[CV 4/5] END .......C=0.1, gamma=1, kernel=poly;, score=0.505 total time=   0.0s
[CV 5/5] END .......C=0.1, gamma=1, kernel=poly;, score=0.081 total time=   0.0s
[CV 1/5] END .....C=0.1, gamma=0.1, kernel=poly;, score=0.347 total time=   0.0s
[CV 2/5] END .....C=0.1, gamma=0.1, kernel=poly;, score=0.563 total time=   0.0s
[CV 3/5] END .....C=0.1, gamma=0.1, kernel=poly;, score=0.210 total time=   0.0s
[CV 4/5] END .....C=0.1, gamma=0.1, kernel=poly;, score=0.647 total time=   0.0s
[CV 5/5] END .....C=0.1, gamma=0.1, kernel=poly;, score=0.168 total time=   0.0s
[CV 1/5] END ...C=0.1, gamma=0.01, kernel=poly;, score=-0.145 total time=   0.0s
[CV 2/5] END ...C=0.1, gamma=0.01, kernel=poly;

,estimator,SVR()
,param_grid,"{'C': [0.1, 1, ...], 'gamma': [1, 0.1, ...], 'kernel': ['poly']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'poly'


In [33]:
grid.best_params_

{'C': 1, 'gamma': 0.1, 'kernel': 'poly'}

In [34]:
grid_pred = grid.predict(X_test)

In [35]:
from sklearn.metrics import mean_absolute_error, r2_score

print("R2 Score:",r2_score(y_test, grid_pred))
print("MAE:",mean_absolute_error(y_test, grid_pred))

R2 Score: 0.5324152087933588
MAE: 4.67009161650049
